<a href="https://colab.research.google.com/github/misamabbas/human-ai-parallel-detection/blob/main/LLM_Detection_01_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets -q

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
np.random.seed(42)

In [ ]:
NUM_SAMPLES_PER_SOURCE_TYPE = 100

In [ ]:
orig_dataset = load_dataset("browndw/human-ai-parallel-corpus")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

text_data/hape-text_gpt-4o-2024-08-06.pa(…):   0%|          | 0.00/17.6M [00:00<?, ?B/s]

text_data/hape-text_gpt-4o-mini-2024-07-(…):   0%|          | 0.00/18.9M [00:00<?, ?B/s]

text_data/hape-text_human-chunk-1.parque(…):   0%|          | 0.00/14.3M [00:00<?, ?B/s]

...

text_data/hape-text_llama-3-70B-Instruct(…):   0%|          | 0.00/12.6M [00:00<?, ?B/s]

text_data/hape-text_llama-3-70B.parquet:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

text_data/hape-text_llama-3-8B-Instruct.(…):   0%|          | 0.00/11.0M [00:00<?, ?B/s]

text_data/hape-text_llama-3-8B.parquet:   0%|          | 0.00/12.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/66320 [00:00<?, ? examples/s]

In [ ]:
# Take a peek at dataset structure, it has only
orig_dataset

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'text'],
        num_rows: 66320
    })
})

In [ ]:
orig_df = pd.DataFrame(orig_dataset['train'])

In [ ]:
def display_full(df):
    with pd.option_context('display.max_rows', None,
                           'display.max_columns', None,
                           'display.max_colwidth', None,
                           'display.width', None):
        display(df)

orig_df.head(2)

,doc_id,text
0,acad_0001@gpt-4o-2024-08-06,"Second, human behavior is inherently variable ..."
1,acad_0002@gpt-4o-2024-08-06,The recent advancements in vehicular safety sy...


In [ ]:
def parse_id_column(df, id_col='doc_id'):
    df[['domain', 'serial_num']] = df[id_col].str.split('_', n=1, expand=True)
    df[['serial_num', 'model']] = df['serial_num'].str.split('@', n=1, expand=True)
    return df

df = parse_id_column(orig_df)
df.head()

,doc_id,text,domain,serialno,model,serial_num
0,acad_0001@gpt-4o-2024-08-06,"Second, human behavior is inherently variable ...",acad,0001,gpt-4o-2024-08-06,0001
1,acad_0002@gpt-4o-2024-08-06,The recent advancements in vehicular safety sy...,acad,0002,gpt-4o-2024-08-06,0002
2,acad_0003@gpt-4o-2024-08-06,The researchers observed that participants in ...,acad,0003,gpt-4o-2024-08-06,0003
3,acad_0004@gpt-4o-2024-08-06,"Instead, observing the model may serve as a fo...",acad,0004,gpt-4o-2024-08-06,0004
4,acad_0005@gpt-4o-2024-08-06,This inherent fascination with the eyes can be...,acad,0005,gpt-4o-2024-08-06,0005


In [ ]:
# Get counts of unique domain and model combinations
source_model_counts = df.groupby(['domain', 'model']).size().reset_index(name='count')
print(source_model_counts.sort_values('count', ascending=False))

   domain                      model  count
32   spok           Meta-Llama-3-70B   1721
33   spok  Meta-Llama-3-70B-Instruct   1721
34   spok            Meta-Llama-3-8B   1721
35   spok   Meta-Llama-3-8B-Instruct   1721
36   spok                    chunk_1   1721
37   spok                    chunk_2   1721
38   spok          gpt-4o-2024-08-06   1721
39   spok     gpt-4o-mini-2024-07-18   1721
15   blog     gpt-4o-mini-2024-07-18   1526
14   blog          gpt-4o-2024-08-06   1526
13   blog                    chunk_2   1526
12   blog                    chunk_1   1526
11   blog   Meta-Llama-3-8B-Instruct   1526
10   blog            Meta-Llama-3-8B   1526
9    blog  Meta-Llama-3-70B-Instruct   1526
8    blog           Meta-Llama-3-70B   1526
16    fic           Meta-Llama-3-70B   1395
17    fic  Meta-Llama-3-70B-Instruct   1395
18    fic            Meta-Llama-3-8B   1395
19    fic   Meta-Llama-3-8B-Instruct   1395
20    fic                    chunk_1   1395
21    fic                    chu

In [ ]:
def sample_groups(df, n):
    sampled_df = pd.DataFrame()
    for source_type in df['domain'].unique():
        source_df = df[df['domain'] == source_type]
        serial_nums = source_df['serial_num'].unique()
        if len(serial_nums) <=n:
          sampled_serial_nums = serial_nums
        else:
          sampled_serial_nums = np.random.choice(serial_nums, size=n, replace=False)

        for serial_num in sampled_serial_nums:
          sampled_df = pd.concat([sampled_df, source_df[source_df['serial_num'] == serial_num]])
    return sampled_df

sample_df = sample_groups(df, NUM_SAMPLES_PER_SOURCE_TYPE)
print(sample_df.groupby("domain").size().reset_index(name='count'))

  domain  count
0   acad    800
1   blog    800
2    fic    800
3   news    800
4   spok    800
5    tvm    800


In [ ]:
def transpose_df(df):
    return df.pivot_table(
        index=['serial_num', 'domain'],
        columns='model',
        values='text',
        aggfunc='first'
    ).reset_index()

# Usage
sample_df_T = transpose_df(sample_df)
print(len(sample_df_T))

600


In [ ]:
sample_df_T.head()

model,serial_num,domain,Meta-Llama-3-70B,Meta-Llama-3-70B-Instruct,Meta-Llama-3-8B,Meta-Llama-3-8B-Instruct,chunk_1,chunk_2,gpt-4o-2024-08-06,gpt-4o-mini-2024-07-18
0,0001,blog,How to consider opposing views and weigh them ...,...and how to question what you're told. That'...,How to take things apart and put them back tog...,"But, I digress. So, as I was saying, this spee...","A few years ago, in 1998 actually, somehow I h...",How to seek out and discover on your own. How ...,What you consume and what you experience. In a...,How to question the world around you and deriv...
1,0002,tvm,Her arms are outstretched. Anita stands and th...,"As she approaches, Anita's eyes widen, her gaz...",She stops behind Anita. You've done well. Very...,Cruella's eyes gleam with a calculating intens...,She's hit and staggers back. She falls off the...,You've done wonderful work for me. Anita nods ...,"She pauses, her silhouette framed against the ...","As she approaches, the air thickens with an un..."
2,0005,fic,He had light hair and blue eyes. He was dresse...,"As he rode beside Steinmetz, his gaze wandered...","His face was thin, but kindly, and it wore a l...","As the sun dipped below the horizon, casting a...","In this country charity covers no sins!"" The s...",He looked like a youthful athlete from Oxford ...,"He rode with a straight-backed earnestness, hi...","His name was Edward Hawthorne, and though he d..."
3,0012,acad,"In adults, similar findings have been reported...",Further research has also explored the role of...,"In addition, experimental manipulations of sem...",The development of childhood fears and anxiety...,Childhood fear and anxiety is highly prevalent...,"In recent years, Approach-Avoidance Tasks (AAT...","On this foundation, there emerges a crucial un...",The understanding of childhood anxiety is furt...
4,0012,blog,"Mmmmmm. Then, we went up to bed. Day One, Mond...","After dinner, we decided to take a stroll alon...","She ordered coffee and I wanted tea, but they ...","After dinner, we decided to take a stroll alon...","Ok, are ya ready for the play-by-play scoop on...","Amy got around midnight, I guess it was. Crazy...","Monday, May 17, 2004 I woke up to the bright V...","After our whimsical late-night breakfast, the ..."


In [ ]:
# Rename columns for ease of use

# Rename the specified columns and drop the others
working_df = sample_df_T.rename(columns={
    'Meta-Llama-3-70B-Instruct': 'llama',
    'gpt-4o-2024-08-06': 'gpt'
})

# Keep only the columns we want
working_df = working_df[['serial_num', 'domain','chunk_1', 'chunk_2','gpt', 'llama']]
print("\nAfter renaming\n")

for col in working_df.columns:
  print(col)


After renaming

serial_num
domain
chunk_1
chunk_2
gpt
llama


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

output_folder = '/content/drive/MyDrive/shared_data'

if not os.path.exists(output_folder):
      os.makedirs(output_folder)

working_df.to_parquet(f'{output_folder}/llm_detection_data.parquet')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
